# Affiner un petit LLM pour l'extraction, avec LoRA

Dans le TP d'extraction, un LLM de 4 milliards de paramètres a extrait les entités de la presse de 1914. Il a aussi étiqueté 2 000 autres passages : ces étiquettes vont apprendre la tâche à un modèle **six fois plus petit**, Qwen3-0.6B, avec LoRA. C'est la distillation : le grand modèle sert une fois, le petit sert en production.

1. Le petit modèle tel quel : que vaut-il ?
2. Préparer les données d'entraînement
3. LoRA : combien de paramètres entraîne-t-on ?
4. Entraîner
5. Le petit modèle affiné face au grand

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et modèle

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/distill.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/teacher-gold.jsonl
%pip install -q peft trl rapidfuzz
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import json
import time

import pandas as pd
import torch
from rapidfuzz import fuzz
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")

gold = pd.read_json("gold.jsonl", lines=True)
distill = pd.read_json("distill.jsonl", lines=True)
teacher = pd.read_json("teacher-gold.jsonl", lines=True)
FIELDS = ["people", "places", "organizations"]
print(len(gold), "passages de test,", len(distill), "passages étiquetés par le grand modèle")

La mesure du TP d'extraction (F1 par champ, correspondance souple), et le prompt donné au grand modèle.

In [ ]:
def normalize(name: str) -> str:
    return " ".join(name.lower().replace("’", "'").split())


def same(a: str, b: str) -> bool:
    a, b = normalize(a), normalize(b)
    return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evaluate(predictions: list[dict[str, list[str]]]) -> pd.Series:
    scores = {}
    for field in FIELDS:
        found = expected = right_found = right_expected = 0
        for prediction, (_, row) in zip(predictions, gold.iterrows()):
            value = prediction.get(field)  # un petit modèle peut répondre n'importe quoi
            predicted = list(dict.fromkeys(p for p in value if isinstance(p, str))) if isinstance(value, list) else []
            truth = row[field]
            found += len(predicted)
            expected += len(truth)
            right_found += sum(any(same(p, t) for t in truth) for p in predicted)
            right_expected += sum(any(same(p, t) for p in predicted) for t in truth)
        precision, recall = right_found / max(found, 1), right_expected / max(expected, 1)
        scores[field] = 2 * precision * recall / max(precision + recall, 1e-9)
    return pd.Series(scores).round(2)


INSTRUCTIONS = """Extrais les entités nommées de ce passage de presse de 1914.
- people : les personnes, sans leur titre (M., Mme, sir, lord) ;
- places : les lieux (pays, régions, villes, rues, bâtiments) ;
- organizations : les organisations nommées (assemblées, partis, syndicats, journaux, agences).
Chaque entité une fois, telle qu'elle est écrite dans le texte, erreurs comprises.
Pas de mentions génériques comme « le gouvernement ». N'invente rien.
Réponds uniquement par un objet JSON avec les clés people, places et organizations."""


def messages(text: str) -> list[dict[str, str]]:
    return [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": text}]


print("Grand modèle :", dict(evaluate(teacher.to_dict("records"))))

## 1. Le petit modèle tel quel

Écrivez `extract_all(texts)` qui génère, par lots de 8, la réponse du modèle à chaque passage (`tokenizer.apply_chat_template(..., add_generation_prompt=True, enable_thinking=False)`, puis `model.generate(..., max_new_tokens=256, do_sample=False)`), et la lit en JSON (`json.loads`, un dictionnaire vide si la réponse n'est pas un JSON valide). Mesurez le F1 sur les 40 passages de test, la part de réponses invalides et le temps par passage.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def extract_all(texts: list[str], batch_size: int = 8) -> tuple[list[dict], int]:
    results, invalid = [], 0
    for start in range(0, len(texts), batch_size):
        prompts = [
            tokenizer.apply_chat_template(messages(t), tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in texts[start : start + batch_size]
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        for output in outputs[:, inputs.input_ids.shape[1] :]:
            answer = tokenizer.decode(output, skip_special_tokens=True).strip()
            answer = answer.removeprefix("```json").removesuffix("```").strip()
            try:
                parsed = json.loads(answer)
                results.append(parsed if isinstance(parsed, dict) else {})
            except json.JSONDecodeError:
                results.append({})
                invalid += 1
    return results, invalid


start = time.time()
base_predictions, invalid = extract_all(gold.text.tolist())
base_seconds = (time.time() - start) / len(gold)
print(f"{invalid} réponses invalides, {base_seconds:.2f} s par passage")
evaluate(base_predictions)

## 2. Préparer les données d'entraînement

Chaque exemple d'entraînement est une conversation : le prompt (consignes et passage), et la réponse attendue, l'objet JSON du grand modèle. Construisez un `datasets.Dataset` aux colonnes `prompt` (la liste des messages) et `completion` (une liste d'un message `assistant`, dont le contenu est `json.dumps(étiquettes, ensure_ascii=False)`). Gardez 100 exemples pour la validation.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from datasets import Dataset


def example(row) -> dict:
    labels = {field: row[field] for field in FIELDS}
    return {
        "prompt": messages(row["text"]),
        "completion": [{"role": "assistant", "content": json.dumps(labels, ensure_ascii=False)}],
    }


dataset = Dataset.from_list([example(row) for row in distill.to_dict("records")])
split = dataset.train_test_split(test_size=100, seed=0)
print(split)
print(split["train"][0]["completion"][0]["content"])

## 3. LoRA

Avec `peft`, ajoutez au modèle des adaptateurs LoRA de rang 16 sur les projections de l'attention (`LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")`). Combien de paramètres sont entraînés, sur combien ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

## 4. Entraîner

Entraînez une époque avec `trl.SFTTrainer` : `SFTConfig(num_train_epochs=1, per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=2e-4, fp16=True, logging_steps=10, eval_strategy="steps", eval_steps=40, max_length=1024, report_to="none", output_dir="lora")`. La perte n'est calculée que sur la réponse (`completion`), pas sur le prompt. Tracez la perte d'entraînement et de validation.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import matplotlib.pyplot as plt
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="lora",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=40,
    max_length=1024,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=split["train"], eval_dataset=split["test"], processing_class=tokenizer)
start = time.time()
trainer.train()
print(f"{(time.time() - start) / 60:.1f} min d'entraînement")

history = pd.DataFrame(trainer.state.log_history)
plt.plot(history.dropna(subset=["loss"]).step, history.dropna(subset=["loss"]).loss, label="entraînement")
plt.plot(history.dropna(subset=["eval_loss"]).step, history.dropna(subset=["eval_loss"]).eval_loss, "o-", label="validation")
plt.xlabel("pas")
plt.ylabel("perte")
plt.legend()
plt.show()

## 5. Le petit modèle affiné face au grand

Relancez l'extraction des 40 passages de test avec le modèle affiné. Comparez, dans un tableau, les F1 du petit modèle avant et après affinage et ceux du grand modèle, avec le temps par passage. L'affinage a-t-il comblé l'écart ? Que gagne-t-on à mettre le petit modèle en production ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Les adaptateurs fusionnés dans les poids : plus de calcul LoRA à part à chaque couche.
model = model.merge_and_unload()
model.eval()
start = time.time()
tuned_predictions, invalid = extract_all(gold.text.tolist())
tuned_seconds = (time.time() - start) / len(gold)
print(f"{invalid} réponses invalides, {tuned_seconds:.2f} s par passage")

pd.DataFrame({
    "Qwen3-0.6B": evaluate(base_predictions),
    "Qwen3-0.6B + LoRA": evaluate(tuned_predictions),
    "grand modèle (4B)": evaluate(teacher.to_dict("records")),
})

# L'affinage comble l'essentiel de l'écart avec le grand modèle, sur cette tâche étroite,
# pour un modèle six fois plus petit : en production, il tient sur une machine bien moins
# chère, ou traite plus de passages à la fois sur le même GPU. Les étiquettes du grand
# modèle ne sont pas parfaites (une personne rangée parmi les organisations…) : l'élève
# apprend aussi ses erreurs.